# 02 — MLP LOLO Evaluation
# Giai đoạn 2 — Mục 2.1, 2.2, 2.4 — Huấn luyện và đánh giá MLP với LOLO
**Đầu ra**:
 - `outputs/tables/mlp_lolo_results.csv`
 - Các model `.h5` và `.tflite` trong `outputs/models/`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from common import models, quantization, training

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Feature

In [3]:
# Chốt cứng 32 đặc trưng: Time (11) + Order (12) + Envelope (9)
feature_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/features_mlp.parquet")

time_cols = [c for c in feature_df.columns if c.startswith("time_")]
order_cols = [c for c in feature_df.columns if c.startswith("order_")]
envelope_cols = [c for c in feature_df.columns if c.startswith("envelope_")]
feature_cols = time_cols + order_cols + envelope_cols

assert len(time_cols) == 11, f"Time features phải có 11, hiện có {len(time_cols)}"
assert len(order_cols) == 12, f"Order features phải có 12, hiện có {len(order_cols)}"
assert len(envelope_cols) == 9, f"Envelope features phải có 9, hiện có {len(envelope_cols)}"
assert len(feature_cols) == 32, f"Tổng số đặc trưng phải đúng 32, hiện có {len(feature_cols)}"

print("Feature breakdown:")
print(f"  Time     : {len(time_cols)}")
print(f"  Order    : {len(order_cols)}")
print(f"  Envelope : {len(envelope_cols)}")
print(f"  TOTAL    : {len(feature_cols)}")
print("\nThứ tự feature_cols:")
for i, c in enumerate(feature_cols, 1):
    print(f"{i:02d}. {c}")


Feature breakdown:
  Time     : 11
  Order    : 12
  Envelope : 9
  TOTAL    : 32

Thứ tự feature_cols:
01. time_mean
02. time_std
03. time_rms
04. time_peak
05. time_kurtosis
06. time_skewness
07. time_variance
08. time_crest_factor
09. time_shape_factor
10. time_impulse_factor
11. time_margin_factor
12. order_f_rot_h1
13. order_f_rot_h2
14. order_f_rot_h3
15. order_BPFO_h1
16. order_BPFO_h2
17. order_BPFO_h3
18. order_BPFI_h1
19. order_BPFI_h2
20. order_BPFI_h3
21. order_BSF_h1
22. order_BSF_h2
23. order_BSF_h3
24. envelope_BPFO_h1
25. envelope_BPFO_h2
26. envelope_BPFO_h3
27. envelope_BPFI_h1
28. envelope_BPFI_h2
29. envelope_BPFI_h3
30. envelope_BSF_h1
31. envelope_BSF_h2
32. envelope_BSF_h3


# MLP - LOLO

### Chuẩn hóa dữ liệu — không dùng StandardScaler toàn cục

StandardScaler phải được `fit` **riêng trên Train của từng LOLO fold** và chỉ `transform` cho Validation/Test. Cách này tránh rò rỉ thống kê (mean/std) từ tải đang được giữ làm Test.

Với vector 32 đặc trưng của MLP, các đặc trưng Envelope cũng nằm trong scaler theo fold; không fit scaler trên toàn bộ dataset trước LOLO.


In [4]:
mlp_results = []
history_rows = []
param_rows = []

for fold_info, train_df, val_df, test_df in training.iterate_lolo_splits(feature_df, load_col='load_hp'):
    print(f"\n--- Fold: {fold_info['fold_name']} ---")

    # StandardScaler chống leakage: fit CHỈ trên Train của fold
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols])
    X_val = scaler.transform(val_df[feature_cols])
    X_test = scaler.transform(test_df[feature_cols])

    y_train = train_df['label'].values
    y_val = val_df['label'].values
    y_test = test_df['label'].values

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)
    y_test_enc = le.transform(y_test)

    model = models.build_mlp(input_dim=len(feature_cols))
    model = models.compile_classifier(model)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train_enc,
        validation_data=(X_val, y_val_enc),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    # Lưu Loss/Accuracy theo từng epoch
    for epoch_idx in range(len(history.history['loss'])):
        history_rows.append({
            'fold': fold_info['fold_name'],
            'epoch': epoch_idx + 1,
            'loss': history.history['loss'][epoch_idx],
            'val_loss': history.history['val_loss'][epoch_idx],
            'accuracy': history.history['accuracy'][epoch_idx],
            'val_accuracy': history.history['val_accuracy'][epoch_idx],
        })

    # Đánh giá Float32
    loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    f1 = f1_score(y_test_enc, y_pred, average='macro')

    # Lượng tử hóa INT8
    tflite_bytes = quantization.quantize_model_int8(model, X_train)
    int8_result = quantization.evaluate_tflite_model(tflite_bytes, X_test, y_test_enc)

    # Phục vụ Bảng 2b: số tham số + kích thước file .tflite
    param_count = int(model.count_params())
    tflite_size_bytes = int(len(tflite_bytes))
    tflite_size_kb = tflite_size_bytes / 1024.0

    mlp_results.append({
        'fold': fold_info['fold_name'],
        'test_load': fold_info['test_load'],
        'float_accuracy': acc,
        'float_f1': f1,
        'int8_accuracy': int8_result['accuracy'],
        'epochs': len(history.history['loss']),
        'best_val_loss': float(np.min(history.history['val_loss'])),
        'best_val_accuracy': float(np.max(history.history['val_accuracy'])),
        'num_parameters': param_count,
        'tflite_size_bytes': tflite_size_bytes,
        'tflite_size_kb': tflite_size_kb
    })

    param_rows.append({
        'model': 'MLP',
        'fold': fold_info['fold_name'],
        'num_parameters': param_count,
        'tflite_size_bytes': tflite_size_bytes,
        'tflite_size_kb': tflite_size_kb
    })

    model.save(MODELS_DIR / f"mlp_{fold_info['fold_name']}.h5")
    with open(MODELS_DIR / f"scaler_mlp_{fold_info['fold_name']}.pkl", 'wb') as f:
        pickle.dump(scaler, f)
    quantization.model_bytes_to_file(tflite_bytes, MODELS_DIR / f"mlp_{fold_info['fold_name']}.tflite")



--- Fold: test_load_0 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpcpgrefik\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpcpgrefik\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpcpgrefik'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1264531161360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265623804688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265623803536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265623802960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265623803344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265623803152: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- Fold: test_load_1 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmprv05rlnc\assets


...

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- Fold: test_load_3 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpaxos4kym\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpaxos4kym\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpaxos4kym'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1265631301072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265631308176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265631312208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265631300112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265631312016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1265631308944: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [5]:
mlp_results_df = pd.DataFrame(mlp_results)
mlp_results_df.to_csv(TABLES_DIR / "mlp_lolo_results.csv", index=False)
mlp_results_df.head()

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs,best_val_loss,best_val_accuracy,num_parameters,tflite_size_bytes,tflite_size_kb
0,test_load_0,0,0.7,0.730952,0.9,100,0.353108,1.000000,1652,5672,5.539062
1,test_load_1,1,1.0,1.000000,1.0,100,0.419096,1.000000,1652,5672,5.539062
2,test_load_2,2,1.0,1.000000,1.0,100,0.265430,0.833333,1652,5672,5.539062
3,test_load_3,3,0.8,0.833333,0.9,100,0.384346,0.833333,1652,5672,5.539062


In [6]:
# Lưu training history chi tiết
history_df = pd.DataFrame(history_rows)
history_df.to_csv(TABLES_DIR / "mlp_training_history_per_epoch.csv", index=False)

# Lưu model size
param_df = pd.DataFrame(param_rows)
param_df.to_csv(TABLES_DIR / "mlp_model_size.csv", index=False)

print("\n=== MLP model size ===")
display(param_df)


=== MLP model size ===


,model,fold,num_parameters,tflite_size_bytes,tflite_size_kb
0,MLP,test_load_0,1652,5672,5.539062
1,MLP,test_load_1,1652,5672,5.539062
2,MLP,test_load_2,1652,5672,5.539062
3,MLP,test_load_3,1652,5672,5.539062


### Outputs phục vụ Giai đoạn 4

- `mlp_lolo_results.csv`: kết quả theo fold.
- `mlp_training_history_per_epoch.csv`: Loss/Accuracy và Validation Loss/Accuracy theo từng epoch.
- `mlp_model_size.csv`: số tham số và kích thước `.tflite` theo fold, phục vụ Bảng 2b.
- `scaler_mlp_<fold>.pkl`: StandardScaler được fit riêng trên Train của từng fold.
